# 03 · Embeddings multimodales (Fashion-CLIP) e índice vectorial
Fase 3 del plan: embeddings de **imagen** y **texto** para todo el catálogo con
[Fashion-CLIP](https://huggingface.co/patrickjohncyh/fashion-clip), índice vectorial y pruebas de recuperación.

**Antes de correr en Kaggle:** *Settings → Accelerator → GPU (T4 o P100)* e *Internet → On*.

**Inputs:** el dataset de la competencia H&M (por las imágenes) + la salida de `01_eda` (*Add Input → Your Work*).

**Salida** (la usan los notebooks 04 y 05):
- `img_emb.npy`, `txt_emb.npy`: embeddings L2-normalizados (float32, 512 dims), una fila por artículo.
- `emb_ids.parquet`: el `article_id` de cada fila (mismo orden en ambos `.npy`).

Solo se incluyen los artículos **con imagen** (se pierden 442 de 105,542): sin imagen no se pueden mostrar en la interfaz.

In [ ]:
import os, glob, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

KAGGLE = os.path.exists("/kaggle/input")
def find(name):
    """Busca un archivo en los Inputs del notebook (Kaggle) o en data/ (local)."""
    hits = glob.glob(f"{'/kaggle/input' if KAGGLE else 'data'}/**/{name}", recursive=True)
    assert hits, f"No encontré {name}: agrega como Input el notebook que lo genera"
    return Path(hits[0])

DATA = Path("/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations")  # imágenes
OUT = Path("/kaggle/working") if KAGGLE else Path("data/03-embeddings-output")
OUT.mkdir(parents=True, exist_ok=True)

def img_path(aid):
    return DATA / "images" / aid[:3] / f"{aid}.jpg"

articles = pd.read_parquet(find("articles.parquet"))
cat = articles[articles.has_image].reset_index(drop=True)
print("Artículos con imagen:", len(cat), "/", len(articles))

## Modelo
Fashion-CLIP es un CLIP ViT-B/32 ajustado con ~800k productos de moda: imagen y texto viven en el mismo espacio,
así que sirve para imagen→imagen, texto→imagen e imagen+texto→imagen.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠️ Sin GPU: las imágenes tardarán horas. Activa GPU en Settings → Accelerator.")

MODEL = "patrickjohncyh/fashion-clip"
model = CLIPModel.from_pretrained(MODEL).to(device).eval()
if device == "cuda":
    model = model.half()
proc = CLIPProcessor.from_pretrained(MODEL)

def as_tensor(out):
    # transformers<5 devuelve un tensor; transformers>=5 un objeto cuyo pooler_output es el embedding proyectado
    return out if torch.is_tensor(out) else out.pooler_output

def prep(img):
    """Imagen PIL → tensor 3x224x224. Rellena a cuadrado con blanco en vez de recortar (no corta la prenda)."""
    img = ImageOps.pad(img.convert("RGB"), (224, 224), color="white")
    return proc.image_processor(img, return_tensors="pt")["pixel_values"][0]

@torch.no_grad()
def embed_pixels(x):
    f = as_tensor(model.get_image_features(pixel_values=x.to(device, model.dtype)))
    return torch.nn.functional.normalize(f.float(), dim=-1).cpu().numpy()

@torch.no_grad()
def embed_texts(texts, bs=512):
    out = []
    for s in range(0, len(texts), bs):
        t = proc.tokenizer(list(texts[s:s + bs]), padding=True, truncation=True,
                           max_length=77, return_tensors="pt").to(device)
        f = as_tensor(model.get_text_features(**t))
        out.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)

class Images(torch.utils.data.Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = Image.open(img_path(self.ids[i]))
        img.draft("RGB", (448, 448))   # decodifica el JPEG a menor resolución: mucho más rápido
        return prep(img)

## Embeddings de texto
Se usa la columna `text` construida en `01_eda` (nombre, tipo, color, apariencia, sección y descripción).
CLIP corta a 77 tokens, así que la descripción larga puede quedar truncada (lo importante va primero).

In [ ]:
t0 = time.time()
txt_emb = embed_texts(cat.text.tolist())
print(txt_emb.shape, f"{time.time() - t0:.0f}s")

## Embeddings de imagen
~15-30 min con GPU en Kaggle. Se calcula una sola vez y se guarda.

In [ ]:
t0 = time.time()
dl = torch.utils.data.DataLoader(Images(cat.article_id.tolist()), batch_size=256,
                                 num_workers=os.cpu_count() if KAGGLE else 0, pin_memory=device == "cuda")
img_emb = np.concatenate([embed_pixels(x) for x in tqdm(dl)])
print(img_emb.shape, f"{time.time() - t0:.0f}s")

In [ ]:
np.save(OUT / "img_emb.npy", img_emb)
np.save(OUT / "txt_emb.npy", txt_emb)
cat[["article_id"]].to_parquet(OUT / "emb_ids.parquet")
print(os.listdir(OUT))

## Índice vectorial
El índice es la propia matriz de embeddings normalizados: buscar = producto punto (= similitud coseno)
y quedarse con los k mayores. Es una búsqueda **exacta**, idéntica a `faiss.IndexFlatIP`.

Decisión: no usamos FAISS. Con ~105k vectores numpy responde una consulta en milisegundos, y en macOS
`faiss` + `torch` en el mismo proceso truena (*segmentation fault* por dos copias de OpenMP), lo que rompería
el demo local de la Fase 7. Un índice aproximado (FAISS IVF/HNSW, Qdrant) solo vale la pena con millones de vectores.

In [ ]:
def knn(Q, E, k, chunk=1024):
    """Top-k de E por producto punto para cada fila de Q → (scores, índices), ordenados de mayor a menor."""
    Q = np.atleast_2d(Q)
    S_all, I_all = [], []
    for s in range(0, len(Q), chunk):   # por bloques para no reventar la RAM
        S = Q[s:s + chunk] @ E.T
        I = np.argpartition(-S, k, axis=1)[:, :k]
        Sk = np.take_along_axis(S, I, 1)
        o = np.argsort(-Sk, axis=1)
        S_all.append(np.take_along_axis(Sk, o, 1)); I_all.append(np.take_along_axis(I, o, 1))
    return np.concatenate(S_all), np.concatenate(I_all)

t0 = time.time(); knn(img_emb[0], img_emb, 10); print(f"Una consulta sobre {len(img_emb)} artículos: {1000*(time.time()-t0):.1f} ms")

## Evaluación con metadatos como ground truth
Para 2,000 artículos al azar buscamos sus 10 vecinos más cercanos y medimos qué fracción comparte
**tipo de prenda** (`product_type_name`) y **color** (`colour_group_name`).

Se excluyen los vecinos con el mismo `product_code`: son el mismo producto en otra variante y
casi idénticos, inflarían la métrica. "Azar" es la probabilidad de coincidir eligiendo un artículo aleatorio.

In [ ]:
ptype = cat.product_type_name.to_numpy()
colour = cat.colour_group_name.to_numpy()
pcode = cat.product_code.to_numpy()

def chance(col):
    p = pd.Series(col).value_counts(normalize=True)
    return (p ** 2).sum()

def neighbours(E, Q, q_ids, k=10):
    """Top-k vecinos de cada consulta, sin artículos del mismo producto que la consulta."""
    _, nn = knn(Q, E, k + 50)
    return np.array([[j for j in row if pcode[j] != pcode[qi]][:k] for qi, row in zip(q_ids, nn)])

def match(q_ids, nn):
    t = ptype[nn] == ptype[q_ids][:, None]
    c = colour[nn] == colour[q_ids][:, None]
    return {"tipo@10": t.mean(), "color@10": c.mean(), "tipo+color@10": (t & c).mean()}

rng = np.random.default_rng(42)
q = rng.choice(len(cat), 2000, replace=False)
eval_rows = {
    "Imagen → imagen": match(q, neighbours(img_emb, img_emb[q], q)),
    "Texto → texto": match(q, neighbours(txt_emb, txt_emb[q], q)),
    "Azar": {"tipo@10": chance(ptype), "color@10": chance(colour),
             "tipo+color@10": chance([f"{a}|{b}" for a, b in zip(ptype, colour)])},
}
pd.DataFrame(eval_rows).T.style.format("{:.3f}")

## Texto → imagen (lo que usará el chat)
Consultas en inglés generadas a partir de las 30 combinaciones (color, tipo) más comunes, p. ej. *"a black trousers"*.
Se mide si las 10 imágenes recuperadas tienen ese tipo y ese color.

In [ ]:
combos = cat.groupby(["colour_group_name", "product_type_name"]).size().nlargest(30).index.tolist()
queries = [f"a photo of a {c.lower()} {t.lower()}" for c, t in combos]
qv = embed_texts(queries)

rows = {}
for name, E in [("Texto → imagen", img_emb), ("Texto → texto del catálogo", txt_emb)]:
    _, nn = knn(qv, E, 10)
    rows[name] = {"tipo@10": np.mean([(ptype[r] == t).mean() for r, (c, t) in zip(nn, combos)]),
                  "color@10": np.mean([(colour[r] == c).mean() for r, (c, t) in zip(nn, combos)])}
eval_rows.update(rows)
pd.DataFrame(rows).T.style.format("{:.3f}")

In [ ]:
res = pd.DataFrame(eval_rows).T
res.to_csv(OUT / "retrieval_eval.csv")
res

## Validación cualitativa
Primera columna: consulta. Siguientes: los 5 vecinos más cercanos (de otros productos).

In [ ]:
def show_rows(rows, titles):
    fig, axes = plt.subplots(len(rows), len(rows[0]), figsize=(2.2 * len(rows[0]), 3.2 * len(rows)))
    for ax_row, ids, ttl in zip(np.atleast_2d(axes), rows, titles):
        for ax, aid, t in zip(ax_row, ids, ttl):
            p = img_path(aid)
            ax.imshow(Image.open(p) if p.exists() else Image.new("RGB", (100, 150), "lightgray"))
            ax.set_title(t, fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()

def label(i):
    return f"{ptype[i]}\n{colour[i]}"

qs = rng.choice(len(cat), 5, replace=False)
nn = neighbours(img_emb, img_emb[qs], qs, k=5)
show_rows([[cat.article_id[i] for i in [qi, *r]] for qi, r in zip(qs, nn)],
          [[("CONSULTA\n" if j == 0 else "") + label(i) for j, i in enumerate([qi, *r])] for qi, r in zip(qs, nn)])

In [ ]:
demo = ["a black leather jacket", "a red summer dress with flowers", "blue denim shorts",
        "white sneakers", "a striped knitted sweater"]
_, nn = knn(embed_texts(demo), img_emb, 6)
show_rows([[cat.article_id[i] for i in r] for r in nn],
          [[(f"«{d}»\n" if j == 0 else "") + label(i) for j, i in enumerate(r)] for d, r in zip(demo, nn)])